# SEC Edgar

Extract SEC Edgar information using python library

https://edgartools.readthedocs.io/en/latest/guides/financial-data/

There are 4 levels of financial data
- Company
- Financials
- Statements
- Filing level

For each 10-K annual filing, we extract the following:
- Report text without tables
- Financial statements

Next, we generate embeddings and insert them into vectorDB.

Lastly, Using the company API, we retrieve key financial facts for the past 3 years and insert them into a graphDB



### EdgarTools Functions

In [1]:
!pip install edgartools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.8 MB/s eta 0:00:00


In [2]:
from edgar import set_identity

# Use your name and email (required by SEC)
set_identity("John Doe john.doe@company.com")

In [3]:
from edgar import *

filings = get_current_filings()

In [4]:
def get_filings_company(ticker, start_year):
  c = Company(ticker)
  filings = c.get_filings()
  annual_reports = filings.filter(form="10-K", date=str(start_year) + "-01-01:")
  quarterly_reports = filings.filter(form="10-Q", date=str(start_year) + "-01-01:")

  return annual_reports, quarterly_reports, c

In [5]:
def extract_statements_from_filing(report):
    # Obtain filing content in markdown or plaintext
    # Not used because we want to exclude tables from the text
    # md = report.markdown()
    # txt = report.text()

    tenk = report.obj()
    doc = tenk.document

    ai_text = doc.text(
        clean=True,                  # Clean and normalize text
        include_tables=False,         # Include table content
        table_max_col_width=500,     # Wide columns, avoid truncation
        # max_length=100000            # Optional: limit total length
    )

    # Get filing date
    filing_date = report.filing_date

    # Parse XBRL data
    xbrl = report.xbrl()

    # Access statements through the user-friendly API
    statements = xbrl.statements

    # Display financial statements
    balance_sheet = statements.balance_sheet()
    income_statement = statements.income_statement()
    cash_flow = statements.cashflow_statement()
    equity = statements.statement_of_equity()
    comprehensive = statements.comprehensive_income()

    return ai_text, filing_date, balance_sheet, income_statement, cash_flow, equity, comprehensive






In [6]:
def extract_financial_facts(ticker, period):
  company = Company(ticker)
  # Use the company object directly
  income_stats = company.income_statement(periods=period, concise_format=True)
  balance_stats = company.balance_sheet(periods=period, concise_format=True)
  cash_stats = company.cashflow_statement(periods=period, concise_format=True)

  # Generate LLM-friendly context
  income_stats_context = income_stats.to_llm_context(
      include_metadata=True,      # Include data quality metrics
      include_hierarchy=False,    # Flatten for simplicity (default)
      flatten_values=True         # Create period-prefixed keys (default)
  )
  balance_stats_context = balance_stats.to_llm_context(
      include_metadata=True,      # Include data quality metrics
      include_hierarchy=False,    # Flatten for simplicity (default)
      flatten_values=True         # Create period-prefixed keys (default)
  )
  cash_stats_context = cash_stats.to_llm_context(
      include_metadata=True,      # Include data quality metrics
      include_hierarchy=False,    # Flatten for simplicity (default)
      flatten_values=True         # Create period-prefixed keys (default)
  )
  return income_stats_context, balance_stats_context, cash_stats_context

### Chunking Functions

In [7]:
import math
import re


def chunk_text_by_paragraph_with_overlap(
    text: str,
    target_chunk_size: int = 1200,
    overlap_ratio: float = 0.12,  # 10–15% recommended
) -> list[str]:
    """
    Chunk text by paragraphs with overlap.

    Parameters
    ----------
    text : str
        Input text
    target_chunk_size : int
        Approximate max characters per chunk
    overlap_ratio : float
        Fraction of chunk size used as overlap (0.10–0.15 recommended)

    Returns
    -------
    List[str] : overlapped chunks
    """

    if not text or not text.strip():
        return []

    # Normalize spacing
    text = re.sub(r"\r\n", "\n", text.strip())

    # Split by paragraph (handles multiple blank lines)
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    overlap_size = int(target_chunk_size * overlap_ratio)

    chunks = []
    current_chunk = []
    current_length = 0

    for para in paragraphs:
        para_len = len(para) + 2  # account for spacing

        if current_length + para_len <= target_chunk_size:
            current_chunk.append(para)
            current_length += para_len
        else:
            # finalize chunk
            chunk_text = "\n\n".join(current_chunk)
            chunks.append(chunk_text)

            # build overlap from end of previous chunk
            overlap_text = chunk_text[-overlap_size:]

            # start new chunk with overlap + current paragraph
            current_chunk = [overlap_text, para]
            current_length = len(overlap_text) + para_len

    # add last chunk
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


### Vector Embedding Functions


In [8]:
!pip install openai tenacity pinecone -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.6 MB/s eta 0:00:00


In [9]:
from google.colab import userdata

In [10]:
from typing import Dict, Any, List
from openai import AzureOpenAI
import time
from tqdm import tqdm

def generate_embeddings_from_list(
    records: List[str],
    batch_size: int = 100,
    delay_seconds: float = 2.5,
) -> List[List[float]]:
    """
    Generate embeddings in batches with rate limiting and progress display.

    Args:
        records: List of chuncks
        batch_size: Number of texts per embedding request
        delay_seconds: Delay between batches (seconds)

    Returns:
        List of embedding vectors aligned with input order
    """
    # Initialize Azure OpenAI client
    client = AzureOpenAI(
          api_key=userdata.get("azure_openai"),
          azure_endpoint=userdata.get("azure_openai_endpoint"),
          api_version="2024-12-01-preview",
    )

    embeddings: List[List[float]] = []

    total_batches = (len(records) + batch_size - 1) // batch_size

    for batch_idx in tqdm(range(total_batches), desc="Generating embeddings"):
        start = batch_idx * batch_size
        end = start + batch_size
        batch_texts = records[start:end]

        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch_texts,
        )

        batch_embeddings = [item.embedding for item in response.data]
        embeddings.extend(batch_embeddings)

        # Rate limiting delay (skip delay after last batch)
        if batch_idx < total_batches - 1:
            time.sleep(delay_seconds)

    return embeddings

In [11]:
from pinecone import Pinecone

def insert_into_pinecone(
    embeddings: List[List[float]],
    metadata_list: List[Dict],
    batch_size: int = 100,
):
    """
    Insert embeddings with metadata into a Pinecone index.

    Args:
        embeddings: List of embedding vectors
        metadata_list: List of metadata dicts (same order as embeddings)
        batch_size: Number of vectors per upsert batch
    """

    if len(embeddings) != len(metadata_list):
        raise ValueError("Embeddings and metadata list must be the same length")

    pc = Pinecone(api_key=userdata.get('pinecone'))
    index = pc.Index("financial-reports")

    vectors = []

    for i, (embedding, metadata) in enumerate(zip(embeddings, metadata_list)):
        vector_id = metadata["company_ticker"] + "_" + str(metadata["fiscal_year"]) + "_" + metadata['document_type'] + "_" + metadata['metadata_type'] + "_" + str(i)

        vectors.append({
            "id": vector_id,
            "values": embedding,
            "metadata": metadata,
        })

    # Batch upsert
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)

    print("Insert embeddings into vector DB complete!")

### Knowledge Graph Functions

In [12]:
!pip install neo4j -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 4.9 MB/s eta 0:00:00


In [20]:
from typing import Tuple, Dict


ALLOWED_DOCUMENT_TYPES = {
    "FORM_10K",
    "FORM_10Q",
    "FORM_8K",
    "ANNUAL_REPORT",
    "EARNINGS_CALL_TRANSCRIPT",
}


def generate_document_core_graph_cypher(
    ticker: str,
    cik: str,
    fiscal_year: int,
    document_type_enum: str,
    document_id:  str,
) -> Tuple[str, Dict]:
    """
    Generates Cypher + params to create core document graph structure.

    Creates:
    - Company
    - FiscalYear
    - Document
    - DocumentType (enum)
    - BELONGS_TO + IS_TYPE relationships

    Returns
    -------
    (cypher_query, params)
    """

    if document_type_enum not in ALLOWED_DOCUMENT_TYPES:
        raise ValueError(
            f"Invalid document_type_enum: {document_type_enum}"
        )

    cypher = """
MERGE (c:Company {
    ticker: $ticker,
    cik: $cik
})

MERGE (fy:FiscalYear {
    year: $fiscal_year
})

MERGE (d:Document {
    document_id: $document_id
})

MERGE (dt:DocumentType {
    name: $document_type
})

MERGE (d)-[:BELONGS_TO]->(c)
MERGE (d)-[:BELONGS_TO]->(fy)
MERGE (d)-[:IS_TYPE]->(dt)
""".strip()

    params = {
        "ticker": ticker,
        "cik": cik,
        "fiscal_year": fiscal_year,
        "document_id": document_id,
        "document_type": document_type_enum,
    }

    return cypher, params

In [21]:
from neo4j import GraphDatabase
import re


def snake_to_pascal(name: str) -> str:
    """
    Convert lower/snake-like metric names to PascalCase.
    Example:
    revenuefromcontractwithcustomerexcludingassessedtax
    -> RevenueFromContractWithCustomerExcludingAssessedTax
    """
    # Insert spaces before capitalizable segments (best effort)
    words = re.findall(r'[a-zA-Z][^A-Z]*', name)
    if not words:
        # fallback: split by underscores if any
        words = name.split("_")
    return "".join(word.capitalize() for word in words)


def insert_financial_facts(
    neo4j_uri: str,
    username: str,
    password: str,
    facts_json: dict,
    company_ticker: str,
    cik: str,
    document_type: str
):
    """
    Inserts financial facts into Neo4j.
    Assumes Company, FiscalYear, Document, DocumentType
    and base relationships already exist.
    """

    driver = GraphDatabase.driver(
        neo4j_uri,
        auth=(username, password)
    )

    with driver.session() as session:

        for key, value in facts_json.items():

            # -----------------------------------
            # 1️⃣ Parse metric name + fiscal year
            # -----------------------------------
            try:
                metric_raw, fy_part = key.split("_fy_")
                fiscal_year = int(fy_part)
            except ValueError:
                print(f"Skipping malformed key: {key}")
                continue

            metric_name = snake_to_pascal(metric_raw)

            # Construct document_id exactly as your schema defines
            document_id = f"{company_ticker}_{fiscal_year}_{document_type}"

            # -----------------------------------
            # 2️⃣ Cypher Query
            # -----------------------------------
            cypher = """
            MATCH (d:Document {document_id: $document_id})

            MERGE (m:Metric {name: $metric_name})

            CREATE (f:Fact {
                value: $value,
                fiscal_year: $fiscal_year,
                company_ticker: $company_ticker
            })

            MERGE (d)-[:REPORTS]->(f)
            MERGE (f)-[:FOR_METRIC]->(m)
            """

            session.run(
                cypher,
                document_id=document_id,
                metric_name=metric_name,
                value=value,
                fiscal_year=fiscal_year,
                company_ticker=company_ticker
            )

    print("Financial facts inserted successfully.")

In [22]:
from neo4j import GraphDatabase


def execute_cypher_transaction(
    cypher_query: str,
    params: Dict,
    neo4j_uri: str,
    username: str,
    password: str,
):
    """
    Executes a Cypher query in a single transaction.
    """

    driver = GraphDatabase.driver(
        neo4j_uri,
        auth=(username, password)
    )

    try:
        with driver.session() as session:
            result = session.execute_write(
                lambda tx: tx.run(cypher_query, **params).consume()
            )
        return result
    finally:
        driver.close()

## Main Code Execution

In [23]:
from datetime import datetime

MAG7 = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA"]

MAG7_CIK = {
    "AAPL": "0000320193",
    "MSFT": "0000789019",
    "GOOGL": "0001652044",
    "AMZN": "0001018724",
    "NVDA": "0001045810",
    "META": "0001326801",
    "TSLA": "0001318605",
}

In [ ]:
for company in MAG7:
  print(company)
  a, q, c = get_filings_company(company, 2023)

  for report in a:
    stmt = extract_statements_from_filing(report)
    filing_text = stmt[0]
    filing_date = stmt[1]
    balance_sheet = stmt[2]
    income_statement = stmt[3]
    cash_flow = stmt[4]
    equity = stmt[5]
    comprehensive = stmt[6]

    # Vectorize Report Text
    print("Chunking Text")
    chunked_text = chunk_text_by_paragraph_with_overlap(
      filing_text,
      1200,
      0.12,  # 10–15% recommended
    )
    print("Generating Embeddings")
    embeddings = generate_embeddings_from_list(chunked_text)
    metadata = [{"text": value} for value in chunked_text]

    for i in metadata:
      i['company_ticker'] = company
      i['fiscal_year'] = filing_date.strftime("%Y")
      i['document_type'] = 'FORM_10K'
      i['metadata_type'] = 'text'

    # Insert into Vector DB
    try:
        insert_into_pinecone(
            embeddings=embeddings,
            metadata_list=metadata,
            batch_size=50,
        )
        print("Pinecone insertion complete")
    except Exception as e:
        print("pinecone insertion failed")
        print(e)

    # Vectorize Statement Tables
    print("Generating Embeddings for Statement Tables")
    statements = [balance_sheet, income_statement, cash_flow, equity, comprehensive]
    statements_metadata = []
    statements_title = []
    for s in statements:
      md = s.render().to_markdown()
      split = md.split("\n\n")
      title = split[0] + " " + split[1]
      statements_title.append(title)
      statements_metadata.append({"table_markdown":md,
                                  "table_description": title,
                                  "company_ticker": company,
                                  "fiscal_year": filing_date.strftime("%Y"),
                                  "document_type": 'FORM_10K',
                                  "metadata_type": 'table'
                                  })

    statements_embeddings = generate_embeddings_from_list(statements_title)
    try:
        insert_into_pinecone(
            embeddings=statements_embeddings,
            metadata_list=statements_metadata,
            batch_size=50,
        )
        print("Pinecone insertion complete")
    except Exception as e:
        print("pinecone insertion failed")
        print(e)






AAPL
Chunking Text
Generating Embeddings


Generating embeddings: 100%|██████████| 2/2 [00:07<00:00,  3.99s/it]


Insert embeddings into vector DB complete!
Pinecone insertion complete
Generating Embeddings for Statement Tables


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.17it/s]


Insert embeddings into vector DB complete!
Pinecone insertion complete
Chunking Text
Generating Embeddings


...

Insert embeddings into vector DB complete!
Pinecone insertion complete
Generating Embeddings for Statement Tables


Generating embeddings: 100%|██████████| 1/1 [00:00<00:00,  1.20it/s]


Insert embeddings into vector DB complete!
Pinecone insertion complete
Chunking Text
Generating Embeddings


Generating embeddings:  67%|██████▋   | 2/3 [00:13<00:06,  6.53s/it]


BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 13130 tokens (13130 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [24]:
# Retrieve Financial Facts and Insert into GraphDB

for company in MAG7:
  cik = MAG7_CIK[company]
  facts = extract_financial_facts(company, 3)
  document_type = "FORM_10K"

  fiscal_years = [int(a[-4:]) for a in facts[0]['periods']]

  for year in fiscal_years:
    print(year)
    core_cypher, core_params = generate_document_core_graph_cypher(
      company,
      cik,
      year,
      document_type,
      str(company) +"_" + str(year) + "_" + document_type,
    )
    try:
      execute_cypher_transaction(
          cypher_query=core_cypher,
          params=core_params,
          neo4j_uri=userdata.get('neo4j_uri'),
          username='neo4j',
          password=userdata.get('neo4j_password'),
      )
      print("Core Graph Insertion Complete")
    except Exception as e:
      print("Core Graph Insertion Failed")
      print(e)

  for fact in facts:
    fact_data = fact['data']
    fact_statement_type = fact['statement_type']
    insert_financial_facts(
      neo4j_uri=userdata.get('neo4j_uri'),
      username='neo4j',
      password=userdata.get('neo4j_password'),
      facts_json=fact_data,
      company_ticker=company,
      cik=cik,
      document_type=document_type
    )


  # not used
  # income_stats = facts[0]
  # balance_stats = facts[1]
  # cash_stats = facts[2]




2025
Core Graph Insertion Complete
2024
Core Graph Insertion Complete
2023
Core Graph Insertion Complete
Financial facts inserted successfully.
Financial facts inserted successfully.
Financial facts inserted successfully.
2025
Core Graph Insertion Complete
2024
Core Graph Insertion Complete
2023
Core Graph Insertion Complete
Financial facts inserted successfully.
Financial facts inserted successfully.
Financial facts inserted successfully.
2025
Core Graph Insertion Complete
2024
Core Graph Insertion Complete
2023
Core Graph Insertion Complete
Financial facts inserted successfully.
Financial facts inserted successfully.
Financial facts inserted successfully.
2025
Core Graph Insertion Complete
2024
Core Graph Insertion Complete
2023
Core Graph Insertion Complete
Financial facts inserted successfully.
Financial facts inserted successfully.
Financial facts inserted successfully.
2026
Core Graph Insertion Complete
2025
Core Graph Insertion Complete
2024
Core Graph Insertion Complete
Financi